# Document Question Answering System using RAG

### Aim

The aim of this project is to build a Retrieval-Augmented Generation (RAG) based document question answering system. The system retrieves relevant information from an uploaded document and generates answers using the Gemini language model.

### Tools Used

- Python
- Google Colab
- Sentence Transformers
- FAISS
- Google Gemini API
- PyPDF
- python-docx

## Installing Dependencies

In [1]:
# Install the core libraries used to build the RAG pipeline
!pip install langchain
!pip install langchain_community
!pip install faiss-cpu
!pip install transformers
!pip install langchain_huggingface
!pip install sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 97.8 MB/s eta 0:00:00


In [2]:
# Install the text splitter package used for chunking documents
!pip install langchain_text_splitters

In [3]:
# Install packages needed to load PDF and DOCX files
!pip install -q langchain langchain-community pypdf docx2txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 11.3 MB/s eta 0:00:00


## Importing Required Libraries
This section imports all the libraries required for building the RAG pipeline.

In [4]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

/tmp/ipykernel_1208/3581772440.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader, Docx2txtLoader


## Document Loading
The user uploads a document that will be used for question answering. The system supports PDF, DOCX and TXT files.

In [5]:
def load_source_document(file_name):
    # Pick the right loader based on the file extension
    if file_name.endswith(".pdf"):
        loader = PyPDFLoader(file_name)
    elif file_name.endswith(".txt"):
        loader = TextLoader(file_name)
    elif file_name.endswith(".docx"):
        loader = Docx2txtLoader(file_name)
    else:
        # Stop early if the extension isn't one we support
        raise ValueError(
            f"Unsupported file extension: {file_name}. Use .txt, .pdf, or .docx"
        )

    # Load the document contents into memory
    return loader.load()

## Text Chunking

The extracted text is divided into smaller chunks. This improves retrieval accuracy by allowing the system to search smaller sections of the document.



In [6]:
def split_into_chunks(document, chunk_size=300, chunk_overlap=52):
    # Break the document into overlapping chunks so retrieval is more accurate
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    doc_chunks = splitter.split_documents(document)

    # Give each chunk an id so it can be referenced later
    for position, chunk in enumerate(doc_chunks):
        chunk.metadata["chunk_id"] = position

    return doc_chunks

## Embedding Model

Each chunk is converted into a numerical representation using a sentence transformer model. These embeddings are later used for semantic search.

In [7]:
# Embedding models used to turn text chunks into vectors
embed_minilm = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L12-v2")
embed_bge_small = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
embed_bge_large = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

## Building the Vector Store

The generated embeddings are stored in a FAISS vector database to enable fast similarity search.

In [8]:
def create_faiss_store(doc_chunks, embedding_model, index_name):
    # Build a FAISS vector store from the chunks
    store = FAISS.from_documents(documents=doc_chunks, embedding=embedding_model)
    # Save it locally so it can be reloaded without rebuilding
    store.save_local(index_name)
    return store

## Loading the Vector Store
We're processing your documents, generating embeddings, and building the semantic index. Once complete, your data will be ready for fast and accurate AI-powered retrieval.

In [9]:
def load_faiss_store(index_name, embedding_model):
    # Reload a previously saved FAISS vector store from disk
    return FAISS.load_local(
        index_name,
        embedding_model,
        allow_dangerous_deserialization=True,
    )

## Running the Pipeline: Load, Chunk, and Embed the Document

In [10]:
# Ask the user for the document to load
file_name = input("Enter the file name:")

document = load_source_document(file_name)
chunks = split_into_chunks(document)
print(f"Loaded document and split into {len(chunks)} chunks\n")

print("Different Embedding Models are loading")

store_minilm = create_faiss_store(chunks, embed_minilm, "faiss_index_minilm")
store_bge_small = create_faiss_store(chunks, embed_bge_small, "faiss_index_bge")
store_bge_large = create_faiss_store(chunks, embed_bge_large, "faiss_index_bge_large")

primary_store = load_faiss_store("faiss_index_minilm", embed_minilm)
print("Vector stores built: faiss_index_minilm (default) and faiss_index_bge, faiss_index_bge_large (for comparison).")

Enter the file name:vs.pdf
Loaded document and split into 46 chunks

Different Embedding Models are loading
Vector stores built: faiss_index_minilm (default) and faiss_index_bge, faiss_index_bge_large (for comparison).


## Loading the Language Model

In [11]:
# Language model used to generate answers from the retrieved context
from langchain_community.llms import HuggingFacePipeline

primary_llm = HuggingFacePipeline.from_model_id(
    model_id="microsoft/Phi-3-mini-4k-instruct",
    task="text-generation",
    device_map="auto",
    pipeline_kwargs={
        "max_new_tokens": 256,
        "do_sample": False,
        "return_full_text": False,
    },
)

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


## Answer Generation Function
Combining retrieved knowledge with language model reasoning to produce a relevant and accurate response.

In [12]:
def build_prompt(context, question):
    # Shared prompt template used by every model we call
    return f"""<|system|>
You are a helpful AI assistant for document question answering.
Use only the information provided in the context below to answer the question.
Rules:
- Answer only from the provided context.
- Do not make up or assume any information.
- If the answer is not found in the context, reply: "I couldn't find that information in the document."
- Keep your answer clear and concise.<|end|>
<|user|>
Context:
{context}

Question:
{question}<|end|>
<|assistant|>
"""


def extract_reply(raw_output):
    # Keep only the assistant's part of the response
    if "<|assistant|>" in raw_output:
        reply = raw_output.split("<|assistant|>")[-1]
    else:
        reply = raw_output

    # Cut off anything the model rambles into after the actual answer
    for stop_token in ["<|end|>", "<|user|>", "<|system|>", "Question:"]:
        if stop_token in reply:
            reply = reply.split(stop_token)[0]

    return reply.strip()


def generate_answer(query, retrieved_docs, model=None):
    model = model or primary_llm
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)
    prompt = build_prompt(context, query)
    raw_output = model.invoke(prompt)
    return extract_reply(raw_output)

## Ask a Question

In [13]:
# Get the user's question
user_query = input("enter the question:")

enter the question:give me the quick summary of this pdf


## Retrieve Relevant Chunks
Searching the vector database to identify document segments that are most relevant to your query. Semantic similarity search is performed on embedded content, followed by ranking and selection of the most contextually meaningful information. The retrieved knowledge is then prepared and organized to provide accurate and informed answer generation

In [14]:
# Retrieve the most relevant chunks for the query
top_chunks = primary_store.similarity_search(user_query, k=5)

print(f"Retrieved {len(top_chunks)} chunks:\n")
# Preview each retrieved chunk
for position, doc in enumerate(top_chunks):
    print(f"--- Chunk {doc.metadata.get('chunk_id')} ---")
    print(doc.page_content[:200], "...\n")

Retrieved 5 chunks:

--- Chunk 45 ---
reference; see the original report for full detail, data sources and the complete reference list. ...

--- Chunk 27 ---
Who is driving this agenda
A mosaic of institutions already work on pieces of this puzzle, including FAO's Commission on Genetic
Resources for Food and Agriculture, CGIAR research centres, UNEP, IUCN, ...

--- Chunk 36 ---
●
Recognise and capitalise smallholder-led financial structures such as SACCOs and credit unions ...

--- Chunk 35 ---
●
Direct bilateral, multilateral and public development bank finance towards a just rural transition
●
Fund Indigenous Peoples and local communities directly, especially through locally led adaptation ...

--- Chunk 24 ---
information tools like the My Farm Trees app
●
Enabling national and international policy, including the International Treaty on Plant Genetic Resources for
Food and Agriculture, alongside conflicting ...



## Generate the Answer
Using the retrieved context and the language model's reasoning capabilities to construct a coherent and accurate response.

In [15]:
# Generate the answer using the retrieved chunks
final_answer = generate_answer(user_query, top_chunks)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


## Display the Answer

In [16]:
# Print the final answer
print(final_answer)

The document discusses the importance of recognizing and supporting smallholder-led financial structures, such as SACCOs and credit unions, to drive a just rural transition. It emphasizes the need for direct funding to Indigenous Peoples and local communities, particularly through locally led adaptation finance. Additionally, the document highlights the role of information tools like the My Farm Trees app and the need to enable national and international policies, including the International Treaty on Plant Genetic Resources for Food and Agriculture. It also mentions the potential conflict between frameworks like UPOV and TRIPS, which can undermine farmers' seed rights.


# Improvements & Experiments

The cells below add the experiments and improvements suggested in the project doc, on top of the pipeline above. Nothing in the existing cells is changed.

In [17]:
# Install packages needed for hybrid search (BM25) and re-ranking (cross-encoder)
!pip install -q rank_bm25 sentence-transformers

## 1. Better chunking strategies

Compare the original fixed-size chunking (300/52) with a token-based splitter and a larger-context splitter.

In [18]:
# Import a token-based text splitter (alternative chunking strategy)
from langchain_text_splitters import TokenTextSplitter

def split_by_tokens(document, chunk_size=256, chunk_overlap=32):
    # Split by token count rather than by character count
    splitter = TokenTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    new_chunks = splitter.split_documents(document)
    for position, chunk in enumerate(new_chunks):
        chunk.metadata["chunk_id"] = position
    return new_chunks

def split_large_chunks(document, chunk_size=600, chunk_overlap=100):
    # Use bigger chunks so more context stays together
    return split_into_chunks(document, chunk_size=chunk_size, chunk_overlap=chunk_overlap)

# Compare how many chunks each strategy produces
strategy_results = {
    "original (300/52)": chunks,
    "token_based (256/32)": split_by_tokens(document),
    "large_context (600/100)": split_large_chunks(document),
}

for name, ch in strategy_results.items():
    print(f"{name}: {len(ch)} chunks")

original (300/52): 46 chunks
token_based (256/32): 11 chunks
large_context (600/100): 24 chunks


## 2. Different embedding models

Already implemented above — the notebook builds and compares three embedding models (`all-MiniLM-L12-v2`, `bge-small-en-v1.5`, `bge-large-en-v1.5`). No changes needed here.

In [19]:
# Compare retrieval results across the three embedding-model vector stores
# (store_minilm, store_bge_small, store_bge_large were built earlier when the document was loaded)

compare_query = input("Enter a question to compare embedding models:")

embedding_stores = {
    "all-MiniLM-L12-v2": store_minilm,
    "bge-small-en-v1.5": store_bge_small,
    "bge-large-en-v1.5": store_bge_large,
}

for model_name, store in embedding_stores.items():
    print(f"=== {model_name} ===")
    results = store.similarity_search(compare_query, k=3)
    for doc in results:
        print(f"--- Chunk {doc.metadata.get('chunk_id')} ---")
        print(doc.page_content[:200], "...\n")
    print()

Enter a question to compare embedding models:give me the quick summary of this pdf
=== all-MiniLM-L12-v2 ===
--- Chunk 45 ---
reference; see the original report for full detail, data sources and the complete reference list. ...

--- Chunk 27 ---
Who is driving this agenda
A mosaic of institutions already work on pieces of this puzzle, including FAO's Commission on Genetic
Resources for Food and Agriculture, CGIAR research centres, UNEP, IUCN, ...

--- Chunk 36 ---
●
Recognise and capitalise smallholder-led financial structures such as SACCOs and credit unions ...


=== bge-small-en-v1.5 ===
--- Chunk 44 ---
beneficiaries.
Condensed from: Macqueen, D, Ducros, A, Núñez del Prado Nieto, I and Williamson, D (2026) Catalysing agrobiodiversity: a
call for differentiated and complementary approaches. IIED, Lond ...

--- Chunk 45 ---
reference; see the original report for full detail, data sources and the complete reference list. ...

--- Chunk 6 ---
billion: about $500 billion of it recoverab

## 3. Hybrid search (keyword + vector)

Combine BM25 keyword search with the existing FAISS vector retriever using an ensemble retriever.

In [20]:
!pip install -q langchain langchain-classic

In [21]:
# BM25 does keyword-based retrieval; EnsembleRetriever combines it with vector search
from langchain_community.retrievers import BM25Retriever

# Try every location EnsembleRetriever has lived in across LangChain versions
try:
    from langchain.retrievers import EnsembleRetriever
except ModuleNotFoundError:
    try:
        from langchain_classic.retrievers import EnsembleRetriever
    except ModuleNotFoundError:
        from langchain_community.retrievers import EnsembleRetriever

def make_hybrid_retriever(doc_chunks, vectorstore, k=5, vector_weight=0.5):
    # Keyword-based retriever
    keyword_retriever = BM25Retriever.from_documents(doc_chunks)
    keyword_retriever.k = k

    # Vector-based retriever
    vector_retriever = vectorstore.as_retriever(search_kwargs={"k": k})

    # Combine both retrievers with adjustable weights
    return EnsembleRetriever(
        retrievers=[keyword_retriever, vector_retriever],
        weights=[1 - vector_weight, vector_weight],
    )

# Build the hybrid retriever using the existing chunks and vector store
combined_retriever = make_hybrid_retriever(chunks, primary_store, k=5)

## 4. Re-ranking for better relevance

Use a cross-encoder to re-score the hybrid retrieval results and keep only the most relevant chunks.

In [22]:
# Cross-encoder used to re-score retrieved chunks for relevance
from sentence_transformers import CrossEncoder

cross_encoder_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank_chunks(query, retrieved_docs, top_k=5):
    # Pair the query with each retrieved chunk
    pairs = [[query, doc.page_content] for doc in retrieved_docs]
    # Score each pair for relevance
    scores = cross_encoder_model.predict(pairs)
    # Sort chunks by relevance score, highest first
    ranked = sorted(zip(scores, retrieved_docs), key=lambda x: x[0], reverse=True)
    return [doc for score, doc in ranked[:top_k]]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

## 5. Experiment with different language models

Load an additional language model alongside the existing Phi-3-mini model so answers can be compared.

In [23]:
# Load a second language model to compare answers against Phi-3-mini
secondary_llm = HuggingFacePipeline.from_model_id(
    model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task="text-generation",
    device_map="auto",
    pipeline_kwargs={
        "max_new_tokens": 256,
        "do_sample": False,
        "return_full_text": False,
    },
)

# generate_answer already accepts a model argument, so it can be reused for any LLM
def build_answer_with_model(query, retrieved_docs, model):
    return generate_answer(query, retrieved_docs, model=model)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

##  hybrid retrieval + re-ranking + comparing LLM answers

In [24]:
final_query = input("Enter the question for the improved pipeline:")

# Retrieve chunks using hybrid search, then re-rank them
hybrid_results = combined_retriever.invoke(final_query)
top_reranked = rerank_chunks(final_query, hybrid_results, top_k=5)

# Compare answers generated by two different language models
answer_from_phi3 = generate_answer(final_query, top_reranked)
answer_from_tinyllama = build_answer_with_model(final_query, top_reranked, secondary_llm)

print("Answer (Phi-3-mini):\n", answer_from_phi3)
print("\nAnswer (TinyLlama):\n", answer_from_tinyllama)

Enter the question for the improved pipeline:give me the context 


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Answer (Phi-3-mini):
 Three entry points, three different mechanisms
Because agrobiodiversity is stewarded across very different contexts — from rare-variety hotspots to Indigenous territories to industrial monocultures — the report argues that no single financing mechanism can cover all the needs. For sociocultural work, it means genuinely respecting Indigenous and community autonomy over land, knowledge, and finance. For market-based work, it means designing incentives robust enough to cover the real transition costs of moving from monoculture to agroecology.

security, climate resilience, ecosystem services, and the biocultural heritage of many Indigenous Peoples and local communities, for whom seeds and crops are bound up with ritual, identity, and cosmovision, not just calories.

The report identifies three complementary approaches, or 'agrobiodiversity catalysts' (ABCs):
1. Biocentric approaches
Focused on conserving rare or endangered crop varieties, breeds, and productive ecosy

## Observation
* The uploaded document was processed successfully and completed without issues.
* Text was accurately extracted from the selected document for further analysis.
* The content was segmented into smaller chunks before generating embeddings to improve retrieval efficiency.
* FAISS effectively identified and returned the most relevant chunks for various user queries.
* Responses were generated using the retrieved document context, ensuring answers remained grounded in the source material rather than relying solely on the language model.


## Conclusion
A Retrieval-Augmented Generation (RAG) based document question-answering system was successfully developed and implemented. The uploaded document was processed, converted into vector embeddings, and stored in a FAISS vector database for efficient semantic search. Relevant document chunks were retrieved based on user queries and provided as context to Gemini, which generated accurate and context-aware responses. This project provided valuable insights into how retrieval mechanisms and large language models can be combined to enhance the accuracy, relevance, and reliability of generated answers.
